# Notebook 18 — Building an HTML Report from JSON

This notebook teaches how to turn the JSON produced by Phase 1 into a
self-contained offline HTML report that can be opened in any browser.

We build three progressively richer HTML files:

| Artifact | Adds |
|---|---|
| `01_metadata_report.html` | File picker, `FileReader`, `JSON.parse`, validation, metadata grid |
| `02_triangle_table_report.html` | HTML triangle table, cumulative/incremental toggle |
| `03_interactive_triangle_report.html` | Heatmap, colour legend, ultimate-estimates table |

**No server, CDN, package, or internet connection is required.**
Each file is fully self-contained (inline CSS and JavaScript only).

Reference files used in this notebook:
- `data/output/teaching_phase1/teaching_export_2022-12.json` — the data source
- `data/output/teaching_phase1/teaching_viewer.html` — the production viewer (for comparison)
- `data/output/teaching_phase1/README.md` — JSON schema documentation


In [2]:
import pathlib, json, sys

_cwd = pathlib.Path.cwd()
project_root = _cwd
# Walk up until we find the directory containing src/ibnr_utils/ and README.md.
# for _ in range(5):
#     if (project_root / 'src' / 'ibnr_utils').exists() and (project_root / 'README.md').exists():
#         break
#     project_root = project_root.parent

if str(project_root / 'src') not in sys.path:
    sys.path.insert(0, str(project_root / 'src'))

OUT_DIR      = project_root / 'data' / 'output' / 'teaching_phase1'
OUT_HTML_DIR = OUT_DIR / 'html_basics'
# Create the output directory if it does not already exist.
OUT_HTML_DIR.mkdir(parents=True, exist_ok=True)

JSON_PATH = OUT_DIR / 'teaching_export_2022-12.json'
print(f'project_root : {project_root}')
print(f'OUT_HTML_DIR : {OUT_HTML_DIR}')
print(f'JSON exists  : {JSON_PATH.exists()}')


project_root : c:\Users\derec\Desktop\Diplomado\diplomado-ml-Dereck\Modulo5\m5t3_sesion3
OUT_HTML_DIR : c:\Users\derec\Desktop\Diplomado\diplomado-ml-Dereck\Modulo5\m5t3_sesion3\data\output\teaching_phase1\html_basics
JSON exists  : True


## Architecture: Python → JSON → HTML

```
Python (ibnr_utils)
  └── TeachingExportConfig + run_teaching_export()
        └── development method calculations
              └── export_to_json() ──► teaching_export_2022-12.json
                                              │
                                  ┌───────────▼───────────┐
                                  │   Browser (no server) │
                                  │  FileReader reads the  │
                                  │  local .json file and  │
                                  │  JSON.parse() decodes  │
                                  │  it into a JS object.  │
                                  │  JavaScript then builds│
                                  │  DOM elements and sets │
                                  │  inline styles.        │
                                  └───────────────────────┘
```

The **JSON file is a contract** between Python and the browser.
Python guarantees the schema (version `"1.0"`, required sections, no NaN/Infinity).
The browser validates the schema before rendering anything.

This separation means you can update the Python calculation without touching the HTML,
or redesign the HTML without touching the Python — as long as the JSON contract is respected.


In [3]:
with open(JSON_PATH) as f:
    doc = json.load(f)

m, a, t = doc['metadata'], doc['assumptions'], doc['triangle']
print(f"schema_version  : {doc['schema_version']}")
print(f"cutoff          : {m['cutoff']}")
print(f"concept         : {m['concept']}")
print(f"method          : {m['method']}")
print(f"selected_average: {a['selected_average']}")
print(f"accident_periods: {len(t['accident_periods'])} (first: {t['accident_periods'][0]})")
print(f"development_ages: {len(t['development_ages'])} (first: {t['development_ages'][0]}, last: {t['development_ages'][-1]})")
print()

# Show the key structure of each section.
for section in ['metadata', 'assumptions', 'triangle', 'development', 'ultimate_records']:
    val = doc[section]
    if isinstance(val, dict):
        print(f"doc['{section}'] keys  : {list(val.keys())}")
    elif isinstance(val, list):
        print(f"doc['{section}']       : list of {len(val)} items")
        if val and isinstance(val[0], dict):
            print(f"  record keys: {list(val[0].keys())}")


schema_version  : 1.0
cutoff          : 2022-12
concept         : Loss Incurred
method          : Development Method
selected_average: VW All
accident_periods: 84 (first: 2016-01)
development_ages: 84 (first: 0, last: 83)

doc['metadata'] keys  : ['cutoff', 'basis', 'concept', 'portfolio', 'source_identifier', 'units', 'method']
doc['assumptions'] keys  : ['selected_average', 'tail_convention', 'factor_selection_note', 'validation_note', 'available_averages', 'age_convention']
doc['triangle'] keys  : ['accident_periods', 'development_ages', 'cumulative', 'incremental', 'observed_mask']
doc['development'] keys  : ['ldf_triangle', 'ata_matrix', 'atu_matrix']
doc['ultimate_records']       : list of 84 items
  record keys: ['accident_period', 'actual_development_period', 'applied_atu_period', 'current_incurred', 'atu', 'percent_developed', 'ultimate', 'additional_development', 'flags']


## HTML, CSS, and JavaScript — three separate responsibilities

| Layer | Role | Lives in |
|---|---|---|
| **HTML** | Structure — headings, paragraphs, `<div>` containers, `<table>` skeletons | `<body>` and `<head>` |
| **CSS** | Presentation — colours, fonts, spacing, grid layout, scroll containers | `<style>` block in `<head>` |
| **JavaScript** | Interaction — reading files, parsing JSON, building table rows, toggling views | `<script>` block before `</body>` |
| **JSON** | Data — the prepared Python results; the browser never recalculates anything | External `.json` file loaded at runtime |

Keeping these responsibilities separate makes the code easier to read and extend.
A CSS change (e.g. a different header colour) never requires touching the JavaScript.
A new data column (add a key to `ultimate_records`) requires a JavaScript change
but no HTML structural change.


## HTML, CSS, and JavaScript elements used in this lesson

The three HTML files this notebook generates share a common set of elements.
The tables below explain each one, what it does inside these specific reports,
and where to find it in the generated files.

---

### HTML — document structure

| Element | What it is | What it does in these reports |
|---|---|---|
| `<!DOCTYPE html>` | Document-type declaration | Tells the browser to parse the file as HTML5. Must be the first line of every `.html` file — without it some browsers fall back to a quirks mode that changes how CSS is applied. |
| `<html lang="en">` | Root element | Wraps the entire page. `lang="en"` lets screen readers and search tools know the content language. |
| `<head>` | Metadata container | Holds everything the browser needs before painting content: character encoding, viewport hint, tab title, and the inline stylesheet. Nothing inside `<head>` is visible on the page. |
| `<meta charset="UTF-8">` | Character-encoding declaration | Ensures the browser reads the file as UTF-8. Required for the em-dash (`—`) and narrow no-break space (` `) characters used in the triangle headers and null-cell display. |
| `<meta name="viewport" …>` | Mobile-display hint | `content="width=device-width, initial-scale=1.0"` prevents mobile browsers from scaling a page designed for a wide screen down to a small viewport. |
| `<title>` | Browser-tab title | Sets the text shown in the browser tab and bookmark: `01 — Metadata Report`, `02 — Triangle Table Report`, `03 — Interactive Triangle Report`. |
| `<style>` | Inline stylesheet | Contains all CSS rules for the page. Placing it inside `<head>` means styles are applied before any content is painted, preventing a flash of unstyled content. |
| `<body>` | Visible-content container | Everything the user sees — headings, the file-picker card, tables, error banner, and output sections — lives here. |
| `<script>` | Inline JavaScript block | Contains all event handlers and rendering functions. Placed immediately before `</body>` so the script runs after all HTML elements it references already exist in the DOM. |

---

### HTML — content and layout elements

| Element | What it is | What it does in these reports |
|---|---|---|
| `<h1>`, `<h2>` | Heading levels 1 and 2 | `<h1>` gives each file its page title. `<h2>` introduces the "Development triangle" and "Ultimate estimates" sections in artifact 03. |
| `<p>` | Paragraph | Short introductory sentence on each page: "Select a `teaching_export_*.json` file to display its metadata." |
| `<div>` | Generic block-level container | Used for the file-picker card (`.picker`), the error banner (`#errBox`), and the output region (`#output`). `<div>` has no visual appearance of its own; CSS and JavaScript control how it looks and whether it is shown. |
| `<span>` | Generic inline container | Used in artifact 03 for the legend colour bar (`#legendBar`) and the range label (`#legendRange`). Unlike `<div>`, `<span>` is inline and sits on the same line as adjacent text. |
| `<label for="…">` | Form label | Associates the "JSON file:" text with the `<input>` element. Clicking the label opens the file picker, increasing the clickable area. |
| `<input type="file">` | File-picker control | Shows a button that opens the OS file chooser. `accept=".json"` filters the dialog to JSON files by default. The `change` event fires when the user selects a file. |
| `<button>` | Clickable button | The "Cumulative" and "Incremental" toggle buttons in artifacts 02 and 03. Each has an `onclick` attribute that calls `setMode()` with the corresponding mode string. |
| `<table>` | Table container | Wraps the development triangle (artifacts 02, 03) and the ultimate-records table (artifact 03). |
| `<thead>` | Table header group | Holds the single header row with column labels. CSS `position: sticky; top: 0` is applied to `<thead> th` elements to pin the row as the user scrolls down through 84 accident periods. |
| `<tbody>` | Table body group | Holds the data rows — one `<tr>` per accident period (84 rows for the triangle; 84 for the ultimate table). |
| `<tr>` | Table row | One row per accident period inside `<tbody>`; one row inside `<thead>` for the column headers. |
| `<th>` | Header cell | Column labels in the triangle (`dev 0` through `dev 83`) and "Accident period" corner cell. Rendered with the dark-blue background colour. Age 0 is the first development period; the ATA/ATU factor matrices start at age 1 (they record transitions *between* consecutive ages, so there is no transition into age 0). |
| `<td>` | Data cell | One cell per `data[i][j]` value. Unobserved cells receive `class="null-cell"` for grey styling. Heat cells in artifact 03 receive an inline `style` attribute with computed `background` and `color` values. |

---

### CSS

| Concept | What it is | What it does in these reports |
|---|---|---|
| **Element selector** (`body`, `.tri`) | Targets all elements matching a tag or class name | `body { font-family: system-ui; }` applies the typeface to the whole page. `.tri { border-collapse: collapse; }` removes the default gap between adjacent table cells. |
| **Class selector** (`.picker`, `.active`, `.null-cell`) | Targets elements whose `class` attribute includes that name | `.active` applies the dark-blue fill to the selected toggle button. `.null-cell` applies light-grey background and centred text to unobserved cells. `.picker` styles the file-picker card with background, border, and rounded corners. |
| **ID selector** (`#errBox`, `#output`, `#triTable`) | Targets the single element with that exact `id` | JavaScript selects `#errBox`, `#output`, `#triTable`, `#metaGrid` etc. with `document.getElementById()` to read or update them. |
| `class=""` attribute | Assigns one or more CSS class names to an element | `<button id="btnC" class="active">` marks the Cumulative button as selected at page load. `<td class="null-cell">` marks an unobserved triangle cell for grey styling. |
| `id=""` attribute | Assigns a unique identifier to one element | Every element referenced by JavaScript receives a unique `id`. `id="fileInput"`, `id="errBox"`, `id="output"`, `id="triTable"` each appear once in the HTML and once in the JavaScript. |
| `color`, `background`, `border` | Text colour, fill colour, border line | `thead th { background: #1a4c8c; color: #fff; }` produces the dark-blue header row. `.error-box { border-left: 4px solid #c0392b; }` draws a red stripe on the error banner. |
| `padding`, `margin` | Inner spacing (inside border), outer spacing (outside border) | `body { margin: 40px auto; padding: 0 24px; }` centres the content block and adds side gutters. `td { padding: 3px 7px; }` keeps cell values from touching their borders. |
| `font-family`, `font-size`, `font-weight` | Typeface, size, and weight | `font-family: system-ui, sans-serif` uses the platform default font. `.meta-label { font-weight: 600; }` bolds the metadata field names. `.tri { font-size: 11px; }` keeps triangle cells compact. |
| `display: grid` + `grid-template-columns` | CSS Grid layout | `.meta-grid { display: grid; grid-template-columns: 200px 1fr; gap: 6px 20px; }` places label divs in a fixed 200 px left column and value divs in a wider right column — without using a `<table>`. |
| `display: flex` | Flexbox single-axis layout | `.picker` and `.toggle-group` use `display: flex` to align their children in a horizontal row with a consistent `gap` between them. |
| `position: sticky` | Keeps an element pinned at the edge of its scroll container | `thead th { position: sticky; top: 0; z-index: 1; }` keeps the header row visible while scrolling down. `tbody td:first-child { position: sticky; left: 0; }` keeps the accident-period label column visible while scrolling right. |
| `overflow: auto`, `max-height` | Creates a scrollable box when content exceeds the size | `.tri-scroll { overflow: auto; max-height: 460px; }` wraps the 84×84 triangle in a scrollable viewport so it does not push the rest of the page off screen. |
| Inline `style="…"` attribute | Overrides a stylesheet rule for a single element | Used in artifact 03 exclusively for per-cell heatmap colours: `<td style="background:rgb(R,G,B);color:#111">`. The values are computed by JavaScript on every `renderTriangle()` call and cannot be defined in a stylesheet because they depend on the data. |

---

### JavaScript

| Concept | What it is | What it does in these reports |
|---|---|---|
| `addEventListener('change', fn)` | Registers a callback for a DOM event | Attached to `#fileInput`. Fires when the user selects a file; the handler creates a `FileReader` and starts reading the file as text. |
| `FileReader` | Built-in browser API for reading local files | `new FileReader()` creates a reader instance. `reader.readAsText(file)` starts an asynchronous read. When the read completes, `reader.onload` fires and `evt.target.result` holds the entire file as a UTF-8 string. No server or HTTP request is involved. |
| `JSON.parse(text)` | Converts a JSON string into a JavaScript object | Turns the file string into `doc`. After a successful parse, `doc.metadata`, `doc.triangle`, `doc.ultimate_records`, etc. are plain JavaScript objects and arrays that the rendering functions can access by property name. Throws `SyntaxError` if the text is not valid JSON. |
| `document.getElementById(id)` | Returns the DOM element with a given `id` | Used before every read or update: `document.getElementById('errBox')` to show the error banner; `document.getElementById('triTable')` to replace the triangle; `document.getElementById('output')` to reveal the output section. |
| `element.style.display` | Sets the inline `display` CSS property of an element | `element.style.display = 'none'` hides the element. `element.style.display = ''` removes the inline override so the stylesheet default (which makes the element visible) takes effect again. |
| `document.createElement(tag)` | Creates a new DOM element in memory | Used in `renderMeta()` to create `<div>` elements for each metadata label and value. The element does not appear on the page until it is added with `appendChild()`. |
| `element.textContent` | Gets or sets the plain-text content of an element | `lDiv.textContent = rows[i][0]` writes the label string into a div. Unlike `innerHTML`, `textContent` never parses HTML or executes scripts, so it is safe even for values that might contain angle brackets. |
| `element.appendChild(child)` | Inserts a child node at the end of an element | Adds each label-div and value-div pair to `#metaGrid` in order, building the metadata grid one row at a time. |
| `element.innerHTML` | Gets or sets the raw HTML content of an element | Used in `renderTriangle()` and `renderUltimate()` to replace the entire table content with a single pre-built HTML string. Assigning one long string is faster than calling `createElement` 7 056 times for the 84×84 triangle. |
| `loadDoc(text)` | Entry-point function for all three reports | Called by the `FileReader.onload` handler. Calls `JSON.parse`, validates the schema version and required sections, then delegates to the appropriate render functions. |
| `renderMeta(doc)` | Metadata-rendering function (artifact 01) | Reads `doc.metadata`, `doc.assumptions`, and `doc.triangle` to build the two-column definition grid in `#metaGrid`. |
| `setMode(mode)` | Toggle-handling function (artifacts 02, 03) | Receives `'cumulative'` or `'incremental'`. Updates `class` on both toggle buttons so only the active one is highlighted, then calls `renderTriangle(mode)`. |
| `renderTriangle(mode)` | Triangle-rendering function (artifacts 02, 03) | Selects `doc.triangle.cumulative` or `.incremental`, iterates rows and columns, and builds the table HTML string. In artifact 03 it also calls `heatRgb()` and `contrastColor()` for each observed cell. |
| `heatRgb(v, vmin, vmax)` | Colour-interpolation function (artifact 03) | Linearly interpolates between RGB anchor colours to produce a background colour for one cell. Positive values shade white-to-blue; negative values (incremental) shade white-to-red. |
| `contrastColor(rgb)` | Text-colour function (artifact 03) | Applies the BT.601 perceptual-luminance formula to the computed background colour and returns `'#111'` (dark) or `'#fff'` (white) so the cell value is always readable. |
| `renderUltimate()` | Ultimate-table function (artifact 03) | Iterates `doc.ultimate_records` and builds the six-column table of reserve quantities using an array of column-accessor functions. |


In [4]:
# The HTML string below is written verbatim to disk.
# It contains all CSS, HTML structure, and JavaScript in one self-contained file.
html_01 = """\
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>01 — Metadata Report</title>
  <style>
    /* Page-wide typography and layout */
    body {
      font-family: system-ui, sans-serif;
      max-width: 860px;
      margin: 40px auto;
      padding: 0 24px;
      color: #222;
      background: #fff;
    }
    h1 { color: #1a4c8c; margin-bottom: 4px; }
    p.intro { margin-top: 4px; color: #555; }

    /* File-picker card */
    .picker {
      display: flex;
      align-items: center;
      gap: 12px;
      margin: 24px 0;
      padding: 14px 16px;
      background: #f0f4fa;
      border-radius: 6px;
      border: 1px solid #d0daea;
    }

    /* Error banner */
    .error-box {
      padding: 10px 14px;
      background: #fdf0f0;
      border-left: 4px solid #c0392b;
      color: #c0392b;
      border-radius: 0 4px 4px 0;
      margin-bottom: 16px;
    }

    /* Two-column definition grid: label | value */
    .meta-grid {
      display: grid;
      grid-template-columns: 200px 1fr;
      gap: 6px 20px;
      margin: 16px 0;
    }
    .meta-label { font-weight: 600; color: #444; }
    .meta-value { font-family: monospace; word-break: break-all; }
  </style>
</head>
<body>
  <h1>01 — Metadata Report</h1>
  <p class="intro">
    Select a <code>teaching_export_*.json</code> file to display its metadata.
    No server is required — the browser reads the local file directly.
  </p>

  <!-- input[type="file"] opens the OS file picker when clicked -->
  <div class="picker">
    <label for="fileInput"><strong>JSON file:</strong></label>
    <input type="file" id="fileInput" accept=".json">
  </div>

  <!-- These areas are hidden until a file is loaded -->
  <div id="errBox" class="error-box" style="display:none"></div>
  <div id="output"                   style="display:none">
    <h2>Report metadata</h2>
    <div id="metaGrid" class="meta-grid"></div>
  </div>

  <script>
    // FileReader reads the selected file as text without any server request.
    document.getElementById('fileInput').addEventListener('change', function (e) {
      var file = e.target.files[0];
      if (!file) return;
      var reader = new FileReader();
      // onload fires when readAsText() finishes; result holds the full file string.
      reader.onload = function (evt) { loadDoc(evt.target.result); };
      reader.readAsText(file);
    });

    function showError(msg) {
      document.getElementById('errBox').textContent = msg;
      document.getElementById('errBox').style.display = '';
      document.getElementById('output').style.display = 'none';
    }

    function loadDoc(text) {
      // JSON.parse() throws SyntaxError if the text is not valid JSON.
      var doc;
      try { doc = JSON.parse(text); }
      catch (e) { showError('Invalid JSON: ' + e.message); return; }

      // Enforce schema version; only "1.0" is supported by this report.
      if (doc.schema_version !== '1.0') {
        showError('Unsupported schema_version: ' + JSON.stringify(doc.schema_version));
        return;
      }

      // Verify every required top-level section exists.
      var required = ['metadata', 'assumptions', 'triangle', 'development', 'ultimate_records'];
      for (var i = 0; i < required.length; i++) {
        if (doc[required[i]] == null) {
          showError('Missing required section: ' + required[i]);
          return;
        }
      }

      // Verify the triangle has the arrays renderMeta() counts.
      var t = doc.triangle;
      if (!Array.isArray(t.accident_periods) || !Array.isArray(t.development_ages)) {
        showError('triangle.accident_periods or triangle.development_ages is missing or not an array');
        return;
      }

      // Wrap the render call so any unexpected data shape shows an error banner.
      try {
        renderMeta(doc);
      } catch (e) {
        showError('Render error: ' + e.message);
      }
    }

    function renderMeta(doc) {
      document.getElementById('errBox').style.display = 'none';
      document.getElementById('output').style.display = '';

      var m = doc.metadata, a = doc.assumptions, t = doc.triangle;

      // Array of [label, value] pairs displayed in the grid.
      var rows = [
        ['Schema version',    doc.schema_version],
        ['Run ID',            doc.run_id],
        ['Generated at',      doc.generated_at],
        ['Cutoff',            m.cutoff],
        ['Concept',           m.concept],
        ['Method',            m.method],
        ['Basis',             m.basis],
        ['Units',             m.units],
        ['Selected average',  a.selected_average],
        ['Tail convention',   a.tail_convention],
        ['Accident periods',  t.accident_periods.length],
        ['Development ages',  t.development_ages.length],
      ];

      // Create a label div and value div for each row, then append to the grid.
      var grid = document.getElementById('metaGrid');
      grid.innerHTML = '';
      for (var i = 0; i < rows.length; i++) {
        var lDiv = document.createElement('div');
        lDiv.className   = 'meta-label';
        lDiv.textContent = rows[i][0];
        var vDiv = document.createElement('div');
        vDiv.className   = 'meta-value';
        // null/undefined fields display as an em-dash
        vDiv.textContent = (rows[i][1] == null) ? '—' : rows[i][1];
        grid.appendChild(lDiv);
        grid.appendChild(vDiv);
      }
    }
  </script>
</body>
</html>
"""

dest = OUT_HTML_DIR / '01_metadata_report.html'
dest.write_text(html_01, encoding='utf-8')
print(f'Written: {dest.name}  ({dest.stat().st_size:,} bytes)')
print('Open in a browser and select teaching_export_2022-12.json to test.')


Written: 01_metadata_report.html  (5,687 bytes)
Open in a browser and select teaching_export_2022-12.json to test.


In [ ]:
# The HTML string below is written verbatim to disk.
# It contains all CSS, HTML structure, and JavaScript in one self-contained file.
with_body = True
with_format = True
with_JSON = True

body = """<body>
  <h1>01 — Metadata Report</h1>
  <p class="intro">
    Select a <code>teaching_export_*.json</code> file to display its metadata.
    No server is required — the browser reads the local file directly.
  </p>

  <!-- input[type="file"] opens the OS file picker when clicked -->
  <div class="picker">
    <label for="fileInput"><strong>JSON file:</strong></label>
    <input type="file" id="fileInput" accept=".json">
  </div>

  <!-- These areas are hidden until a file is loaded -->
  <div id="errBox" class="error-box" style="display:none"></div>
  <div id="output"                   style="display:none">
    <h2>Report metadata</h2>
    <div id="metaGrid" class="meta-grid"></div>
  </div>
<body>
"""

html_01 = """\
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>01 — Metadata Report</title>
  <style>
    /* Page-wide typography and layout */
    body {
      font-family: system-ui, sans-serif;
      max-width: 860px;
      margin: 40px auto;
      padding: 0 24px;
      color: #222;
      background: #fff;
    }
    h1 { color: #1a4c8c; margin-bottom: 4px; }
    p.intro { margin-top: 4px; color: #555; }

    /* File-picker card */
    .picker {
      display: flex;
      align-items: center;
      gap: 12px;
      margin: 24px 0;
      padding: 14px 16px;
      background: #f0f4fa;
      border-radius: 6px;
      border: 1px solid #d0daea;
    }

    /* Error banner */
    .error-box {
      padding: 10px 14px;
      background: #fdf0f0;
      border-left: 4px solid #c0392b;
      color: #c0392b;
      border-radius: 0 4px 4px 0;
      margin-bottom: 16px;
    }

    /* Two-column definition grid: label | value */
    .meta-grid {
      display: grid;
      grid-template-columns: 200px 1fr;
      gap: 6px 20px;
      margin: 16px 0;
    }
    .meta-label { font-weight: 600; color: #444; }
    .meta-value { font-family: monospace; word-break: break-all; }
  </style>
</head>
<body>
# En body venia todo el texto de la página 

  <script>
  ## Muestra los resultados
      }
    }
  </script>
</body>
</html>
"""

html_01 = """\
<!DOCTYPE html>
<html lang="en">
<head>


</html>
"""

dest = OUT_HTML_DIR / '01_metadata_report_pruebas.html'
dest.write_text(html_01, encoding='utf-8')
print(f'Written: {dest.name}  ({dest.stat().st_size:,} bytes)')
print('Open in a browser and select teaching_export_2022-12.json to test.')

Written: 01_metadata_report_pruebas.html  (63 bytes)
Open in a browser and select teaching_export_2022-12.json to test.


In [16]:

import json

# ============================================================
# 0. INTERRUPTORES DEL EXPERIMENTO
# ============================================================

with_body = True
with_format = False
with_JS = True
with_JSON = False


# ============================================================
# 1. CSS — FORMATO VISUAL
# ============================================================

css = """
<style>
  /* Configuración general de la página */
  body {
    font-family: system-ui, sans-serif;
    max-width: 860px;
    margin: 40px auto;
    padding: 0 24px;
    color: #222;
    background: #fff;
  }

  /* Título y descripción */
  h1 {
    color: #1a4c8c;
    margin-bottom: 4px;
  }

  p.intro {
    margin-top: 4px;
    color: #555;
  }

  /* Contenedor para seleccionar el archivo */
  .picker {
    display: flex;
    align-items: center;
    gap: 12px;
    margin: 24px 0;
    padding: 14px 16px;
    background: #f0f4fa;
    border-radius: 6px;
    border: 1px solid #d0daea;
  }

  /* Mensajes de error */
  .error-box {
    padding: 10px 14px;
    background: #fdf0f0;
    border-left: 4px solid #c0392b;
    color: #c0392b;
    border-radius: 0 4px 4px 0;
    margin-bottom: 16px;
  }

  /* Cuadrícula de metadatos */
  .meta-grid {
    display: grid;
    grid-template-columns: 200px 1fr;
    gap: 6px 20px;
    margin: 16px 0;
  }

  .meta-label {
    font-weight: 600;
    color: #444;
  }

  .meta-value {
    font-family: monospace;
    word-break: break-all;
  }
</style>
"""


# ============================================================
# 2. HTML — CUERPO DE LA PÁGINA
# ============================================================

body = """
<h1>01 — Metadata Report</h1>

<p class="intro">
  Select a <code>teaching_export_*.json</code> file
  to display its metadata.

  No server is required.
</p>

<!-- Selector de archivo -->
<div class="picker">
  <label for="fileInput">
    <strong>JSON file:</strong>
  </label>

  <input
    type="file"
    id="fileInput"
    accept=".json"
  >
</div>

<!-- Contenedor de errores -->
<div
  id="errBox"
  class="error-box"
  style="display:none">
</div>

<!-- Contenedor de resultados -->
<div
  id="output"
  style="display:none">

  <h2>Report metadata</h2>

  <div
    id="metaGrid"
    class="meta-grid">
  </div>

</div>
"""


# ============================================================
# 3. JAVASCRIPT — COMPORTAMIENTO
# ============================================================

js = """
<script>

  // ---------------------------------------------------------
  // A. LEER ARCHIVOS DESDE EL SELECTOR
  // ---------------------------------------------------------

  var input = document.getElementById("fileInput");

  if (input) {

    input.addEventListener("change", function(e) {

      var file = e.target.files[0];

      if (!file) return;

      var reader = new FileReader();

      reader.onload = function(evt) {
        loadDoc(evt.target.result);
      };

      reader.onerror = function() {
        showError("Could not read the selected file.");
      };

      reader.readAsText(file);

    });

  }


  // ---------------------------------------------------------
  // B. MOSTRAR ERRORES
  // ---------------------------------------------------------

  function showError(msg) {

    var error = document.getElementById("errBox");
    var output = document.getElementById("output");

    if (!error || !output) return;

    error.textContent = msg;
    error.style.display = "";
    output.style.display = "none";

  }


  // ---------------------------------------------------------
  // C. INTERPRETAR Y VALIDAR EL JSON
  // ---------------------------------------------------------

  function loadDoc(text) {

    var doc;

    // Convertir texto JSON en objeto JavaScript
    try {

      doc = JSON.parse(text);

    } catch(e) {

      showError("Invalid JSON: " + e.message);
      return;

    }

    // Validar versión
    if (doc === null || doc.schema_version !== "1.0") {

      showError("Unsupported schema_version");
      return;

    }

    // Validar secciones principales
    var required = [
      "metadata",
      "assumptions",
      "triangle",
      "development",
      "ultimate_records"
    ];

    for (var i = 0; i < required.length; i++) {

      if (doc[required[i]] == null) {

        showError("Missing section: " + required[i]);
        return;

      }

    }

    // Validar dimensiones del triángulo
    var t = doc.triangle;

    if (
      !Array.isArray(t.accident_periods) ||
      !Array.isArray(t.development_ages)
    ) {

      showError("Invalid triangle structure");
      return;

    }

    // Construir la visualización
    try {

      renderMeta(doc);

    } catch(e) {

      showError("Render error: " + e.message);

    }

  }


  // ---------------------------------------------------------
  // D. CONSTRUIR LA TABLA DE METADATOS
  // ---------------------------------------------------------

  function renderMeta(doc) {

    var grid = document.getElementById("metaGrid");

    if (!grid) return;

    var m = doc.metadata;
    var a = doc.assumptions;
    var t = doc.triangle;

    var rows = [

      ["Schema version", doc.schema_version],
      ["Run ID", doc.run_id],
      ["Generated at", doc.generated_at],
      ["Cutoff", m.cutoff],
      ["Concept", m.concept],
      ["Method", m.method],
      ["Basis", m.basis],
      ["Units", m.units],
      ["Selected average", a.selected_average],
      ["Tail convention", a.tail_convention],
      ["Accident periods", t.accident_periods.length],
      ["Development ages", t.development_ages.length]

    ];

    // Limpiar resultados anteriores
    grid.innerHTML = "";

    // Crear las filas
    for (var i = 0; i < rows.length; i++) {

      var label = document.createElement("div");
      label.className = "meta-label";
      label.textContent = rows[i][0];

      var value = document.createElement("div");
      value.className = "meta-value";

      value.textContent = (
        rows[i][1] == null
      ) ? "—" : rows[i][1];

      grid.appendChild(label);
      grid.appendChild(value);

    }

    // Mostrar resultados y ocultar errores
    document.getElementById("errBox").style.display = "none";
    document.getElementById("output").style.display = "";

  }


  // ---------------------------------------------------------
  // E. CARGAR JSON INCRUSTADO (SI EXISTE)
  // ---------------------------------------------------------

  var embedded = document.getElementById("embeddedJSON");

  if (embedded && document.getElementById("metaGrid")) {

    loadDoc(embedded.textContent);

  }

</script>
"""


# ============================================================
# 4. JSON — DATOS DEL REPORTE
# ============================================================

json_block = ""

if with_JSON:

    # Leemos el archivo ya generado por el proyecto.
    with open(JSON_PATH, "r", encoding="utf-8") as f:
        data = json.load(f)

    # Convertimos el objeto en texto JSON.
    # Escapamos "<" para impedir cierres HTML accidentales.
    json_text = json.dumps(
        data,
        ensure_ascii=False
    ).replace("<", "\\u003c")

    # Lo incorporamos como datos, no como código JavaScript.
    json_block = (
        '<script type="application/json" id="embeddedJSON">\n'
        + json_text
        + '\n</script>\n'
    )


# ============================================================
# 5. CONSTRUCCIÓN DEL DOCUMENTO
# ============================================================

html_parts = [

    # Estructura principal
    "<!DOCTYPE html>\n",
    '<html lang="en">\n',
    "<head>\n",
    '<meta charset="UTF-8">\n',
    '<meta name="viewport" content="width=device-width, initial-scale=1.0">\n',
    "<title>Metadata Report - Pruebas</title>\n",

    # CSS opcional
    css if with_format else "",

    "</head>\n",
    "<body>\n",

    # Contenido HTML opcional
    body if with_body else "",

    # Datos JSON opcionales
    json_block,

    # JavaScript opcional
    js if with_JS else "",

    "</body>\n",
    "</html>"

]

# Concatenar las partes
html_01 = "".join(html_parts)


# ============================================================
# 6. GUARDAR ARCHIVO
# ============================================================

dest = OUT_HTML_DIR / "01_metadata_report_pruebas.html"

dest.write_text(
    html_01,
    encoding="utf-8"
)

print(f"Written: {dest.name}")
print(f"Size: {dest.stat().st_size:,} bytes")

print("\nConfiguración:")
print(f"  HTML:       {with_body}")
print(f"  CSS:        {with_format}")
print(f"  JavaScript: {with_JS}")
print(f"  JSON:       {with_JSON}")


Written: 01_metadata_report_pruebas.html
Size: 5,263 bytes

Configuración:
  HTML:       True
  CSS:        False
  JavaScript: True
  JSON:       False


#### Uso de boton

boton_html = """"""

In [20]:
## Uso de boton

boton_html= """

<!DOCTYPE html>
<html lang="es">

<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">

  <title>Laboratorio de Experimentos Inútiles</title>

  <style>
    * {
      box-sizing: border-box;
    }

    body {
      margin: 0;
      min-height: 100vh;
      padding: 30px 15px;
      background: #0f172a;
      color: white;
      font-family: Arial, sans-serif;
      display: flex;
      justify-content: center;
      align-items: center;
    }

    .contenedor {
      width: 100%;
      max-width: 700px;
      background: #1e293b;
      padding: 30px;
      border-radius: 20px;
      text-align: center;
      box-shadow: 0 15px 50px #0005;
    }

    h1 {
      color: #38bdf8;
      margin-bottom: 10px;
    }

    h2 {
      color: #f8fafc;
    }

    p {
      color: #cbd5e1;
      line-height: 1.6;
    }

    .experimento {
      margin-top: 28px;
      padding: 20px;
      background: #111c30;
      border-radius: 14px;
    }

    button {
      border: none;
      padding: 14px 22px;
      border-radius: 10px;
      font-size: 16px;
      font-weight: bold;
      cursor: pointer;
      transition: transform 0.15s;
    }

    button:disabled {
      opacity: 0.6;
      cursor: not-allowed;
    }

    /* =====================================
       EXPERIMENTO 1: BOTÓN ESCURRIDIZO
       ===================================== */

    .zona-boton {
      height: 200px;
      background: #020617;
      border: 2px dashed #475569;
      border-radius: 12px;
      position: relative;
      overflow: hidden;
    }

    #botonFalso {
      position: absolute;
      left: 30%;
      top: 40%;
      background: #ef4444;
      color: white;
      white-space: nowrap;
      transition: left 0.12s, top 0.12s;
      touch-action: manipulation;
    }

    /* =====================================
       EXPERIMENTO 2: CALCULADORA
       ===================================== */

    #botonCalcular {
      background: #22c55e;
      color: #052e16;
    }

    #botonCalcular:hover:not(:disabled) {
      transform: scale(1.05);
    }

    .barra-contenedor {
      width: 100%;
      height: 22px;
      background: #334155;
      border-radius: 20px;
      margin-top: 25px;
      overflow: hidden;
      display: none;
    }

    #barra {
      width: 0%;
      height: 100%;
      background: linear-gradient(90deg, #38bdf8, #22c55e);
      transition: width 0.2s;
    }

    #estado {
      color: #facc15;
      min-height: 35px;
      margin-top: 20px;
    }

    #resultado {
      font-size: 35px;
      font-weight: bold;
      color: #4ade80;
      margin-top: 10px;
    }

    /* =====================================
       EXPERIMENTO 3: BOTÓN PROHIBIDO
       ===================================== */

    #botonProhibido {
      background: #facc15;
      color: #451a03;
      margin: 15px 0;
    }

    #botonProhibido:hover:not(:disabled) {
      transform: scale(1.12);
      background: #fb923c;
    }

    .contador {
      font-size: 17px;
      color: #facc15;
      font-weight: bold;
    }

    #mensajeCaos {
      color: #fb7185;
      font-size: 19px;
      font-weight: bold;
      min-height: 45px;
    }

    /* =====================================
       EFECTOS ESPECIALES
       ===================================== */

    .personaje {
      position: fixed;
      top: -80px;
      font-size: 45px;
      pointer-events: none;
      z-index: 9999;
      animation: caer linear forwards;
    }

    @keyframes caer {
      0% {
        transform: translateY(0) rotate(0deg);
      }

      100% {
        transform: translateY(115vh) rotate(720deg);
      }
    }

    /* Terremoto */
    .temblor {
      animation: terremoto 0.15s infinite;
    }

    @keyframes terremoto {
      0%   { transform: translate(0, 0); }
      25%  { transform: translate(4px, -3px); }
      50%  { transform: translate(-4px, 3px); }
      75%  { transform: translate(3px, 4px); }
      100% { transform: translate(0, 0); }
    }

    /* Explosión de colores */
    .arcoiris {
      animation: coloresLocos 0.6s infinite alternate;
    }

    @keyframes coloresLocos {
      0%   { background: #0f172a; }
      25%  { background: #581c87; }
      50%  { background: #9f1239; }
      75%  { background: #164e63; }
      100% { background: #14532d; }
    }

    /* Mensaje gigante */
    .mensaje-gigante {
      position: fixed;
      inset: 0;
      display: flex;
      align-items: center;
      justify-content: center;
      padding: 25px;
      z-index: 10000;
      pointer-events: none;
      font-size: clamp(26px, 6vw, 65px);
      font-weight: 900;
      text-align: center;
      color: #facc15;
      text-shadow: 4px 4px 0 #7f1d1d,
                   0 0 20px #000;
      animation: aparecerGigante 2.5s forwards;
    }

    @keyframes aparecerGigante {
      0% {
        opacity: 0;
        transform: scale(0.2) rotate(-15deg);
      }

      30% {
        opacity: 1;
        transform: scale(1.1) rotate(5deg);
      }

      70% {
        opacity: 1;
        transform: scale(1);
      }

      100% {
        opacity: 0;
        transform: scale(1.5);
      }
    }

    @media (prefers-reduced-motion: reduce) {
      .temblor,
      .arcoiris {
        animation: none;
      }

      .personaje {
        animation-duration: 0.5s !important;
        animation-delay: 0s !important;
      }
    }
  </style>
</head>

<body>

  <main class="contenedor" id="contenedor">

    <h1>🧪 Laboratorio de Experimentos Inútiles</h1>

    <p>
      Bienvenido al centro de investigación científica
      menos confiable del planeta.
    </p>

    <!-- =====================================
         EXPERIMENTO 1
         ===================================== -->

    <section class="experimento">

      <h2>😈 Experimento 1: Atrápame</h2>

      <p>Intenta presionar el botón. Buena suerte.</p>

      <div class="zona-boton" id="zona">
        <button id="botonFalso" type="button">
          ¡Presióname!
        </button>
      </div>

    </section>

    <!-- =====================================
         EXPERIMENTO 2
         ===================================== -->

    <section class="experimento">

      <h2>🧠 Experimento 2: Matemáticas extremas</h2>

      <p>
        Nuestra supercomputadora resolverá
        la operación más difícil de la historia.
      </p>

      <button id="botonCalcular" type="button">
        Calcular 2 + 2
      </button>

      <div class="barra-contenedor" id="progreso">
        <div id="barra"></div>
      </div>

      <p id="estado" aria-live="polite"></p>

      <div id="resultado" aria-live="polite"></div>

    </section>

    <!-- =====================================
         EXPERIMENTO 3
         ===================================== -->

    <section class="experimento">

      <h2>☢️ Experimento 3: Botón prohibido</h2>

      <p>
        Este botón fue prohibido por el
        Consejo Internacional de Patos.
      </p>

      <p>
        Cada vez que lo presiones,
        las consecuencias serán peores.
      </p>

      <button id="botonProhibido" type="button">
        🚨 NO PRESIONAR
      </button>

      <p class="contador">
        Veces presionado:
        <span id="contadorClicks">0</span>
      </p>

      <p id="mensajeCaos" aria-live="polite">
        Todavía puedes salvarte...
      </p>

    </section>

  </main>

  <script>

    // ==========================================
    // EXPERIMENTO 1: BOTÓN QUE ESCAPA
    // ==========================================

    const botonFalso = document.getElementById("botonFalso");
    const zona = document.getElementById("zona");

    function moverBoton() {

      // Calcular espacio disponible
      const maxX = Math.max(
        0,
        zona.clientWidth - botonFalso.offsetWidth - 10
      );

      const maxY = Math.max(
        0,
        zona.clientHeight - botonFalso.offsetHeight - 10
      );

      // Elegir coordenadas aleatorias
      const x = Math.random() * maxX;
      const y = Math.random() * maxY;

      // Actualizar posición
      botonFalso.style.left = `${x}px`;
      botonFalso.style.top = `${y}px`;
    }

    botonFalso.addEventListener("pointerenter", moverBoton);

    botonFalso.addEventListener("pointerdown", (evento) => {
      evento.preventDefault();
      moverBoton();
    });

    botonFalso.addEventListener("click", (evento) => {
      evento.preventDefault();
      moverBoton();
    });


    // ==========================================
    // EXPERIMENTO 2: CÁLCULO ULTRACOMPLEJO
    // ==========================================

    const botonCalcular = document.getElementById("botonCalcular");
    const progreso = document.getElementById("progreso");
    const barra = document.getElementById("barra");
    const estado = document.getElementById("estado");
    const resultado = document.getElementById("resultado");

    const mensajes = [
      "Inicializando procesador cuántico...",
      "Consultando a los matemáticos de Harvard...",
      "Verificando que el número 2 exista...",
      "Calculando la masa del universo...",
      "Ejecutando simulaciones intergalácticas...",
      "Preguntándole a ChatGPT...",
      "ChatGPT está confundido...",
      "Descartando que el resultado sea 5...",
      "Solicitando autorización a la NASA...",
      "Preparando descubrimiento revolucionario..."
    ];

    botonCalcular.addEventListener("click", () => {

      botonCalcular.disabled = true;

      progreso.style.display = "block";
      barra.style.width = "0%";
      resultado.textContent = "";

      let porcentaje = 0;

      const intervalo = setInterval(() => {

        porcentaje += 2;

        barra.style.width = `${porcentaje}%`;

        const indice = Math.min(
          Math.floor(porcentaje / 10),
          mensajes.length - 1
        );

        estado.textContent =
          `${mensajes[indice]} (${porcentaje}%)`;

        if (porcentaje >= 100) {

          clearInterval(intervalo);

          // Increíble descubrimiento matemático
          const respuesta = 2 + 2;

          estado.textContent =
            "✅ Cálculo científico completado.";

          resultado.textContent =
            `2 + 2 = ${respuesta} 🎉`;

          botonCalcular.disabled = false;

          botonCalcular.textContent =
            "Repetir investigación científica";
        }

      }, 110);

    });


    // ==========================================
    // EXPERIMENTO 3: CAOS PROGRESIVO
    // ==========================================

    const botonProhibido =
      document.getElementById("botonProhibido");

    const mensajeCaos =
      document.getElementById("mensajeCaos");

    const contadorClicks =
      document.getElementById("contadorClicks");

    const contenedor =
      document.getElementById("contenedor");


    // ------------------------------------------
    // CONTADOR PERSISTENTE
    // ------------------------------------------

    // localStorage conserva el contador aunque
    // se cierre o actualice la página.
    // Es un contador local a este navegador.

    let vecesPresionado = 0;

    try {
      vecesPresionado =
        Number(localStorage.getItem("contadorCaos")) || 0;
    } catch (error) {
      // Si el almacenamiento está bloqueado,
      // el contador funciona durante la sesión.
    }

    contadorClicks.textContent = vecesPresionado;


    // ------------------------------------------
    // CREAR LLUVIA DE PERSONAJES
    // ------------------------------------------

    function crearLluvia(cantidad, personajes) {

      for (let i = 0; i < cantidad; i++) {

        const elemento = document.createElement("div");

        elemento.className = "personaje";

        // Seleccionar un personaje aleatorio
        const indice = Math.floor(
          Math.random() * personajes.length
        );

        elemento.textContent = personajes[indice];

        // Posición horizontal aleatoria
        elemento.style.left =
          Math.random() * 95 + "vw";

        // Retraso aleatorio
        elemento.style.animationDelay =
          Math.random() * 1.2 + "s";

        // Velocidad aleatoria
        elemento.style.animationDuration =
          2 + Math.random() * 2 + "s";

        document.body.appendChild(elemento);

        // Eliminar el personaje después de caer
        setTimeout(() => {
          elemento.remove();
        }, 6500);
      }
    }


    // ------------------------------------------
    // MOSTRAR MENSAJE GIGANTE
    // ------------------------------------------

    function mostrarMensajeGigante(texto) {

      const mensaje = document.createElement("div");

      mensaje.className = "mensaje-gigante";

      mensaje.textContent = texto;

      document.body.appendChild(mensaje);

      setTimeout(() => {
        mensaje.remove();
      }, 2600);
    }


    // ------------------------------------------
    // ACTIVAR DESASTRE SEGÚN EL NIVEL
    // ------------------------------------------

    function activarDesastre(nivel) {

      // Limpiar efectos anteriores
      contenedor.classList.remove("temblor");
      document.body.classList.remove("arcoiris");

      let cantidad;
      let personajes;
      let mensaje;

      if (nivel === 1) {

        // NIVEL 1: INVASIÓN DE PATOS

        cantidad = 15;

        personajes = ["🦆"];

        mensaje = "🦆 Has despertado a los patos.";

      } else if (nivel === 2) {

        // NIVEL 2: PATOS Y GALLINAS

        cantidad = 30;

        personajes = ["🦆", "🐔", "🐥"];

        mensaje =
          "🐔 ¡Los patos solicitaron refuerzos!";

      } else if (nivel === 3) {

        // NIVEL 3: TERREMOTO

        cantidad = 45;

        personajes = ["🦆", "🐔", "🐥", "🐸"];

        contenedor.classList.add("temblor");

        mensaje =
          "🌋 ¡TERREMOTO! ¡LAS RANAS SE UNIERON!";

      } else if (nivel === 4) {

        // NIVEL 4: EXPLOSIÓN DE COLORES

        cantidad = 60;

        personajes = [
          "🦆", "🐔", "🐸", "🦄", "🌈", "💥"
        ];

        contenedor.classList.add("temblor");
        document.body.classList.add("arcoiris");

        mensaje =
          "🌈 ¡HAS ROTO LAS LEYES DE LA FÍSICA!";

      } else if (nivel === 5) {

        // NIVEL 5: ADVERTENCIA SUPREMA

        cantidad = 75;

        personajes = [
          "🦆", "🐔", "🦄", "👽", "💥", "🌈"
        ];

        contenedor.classList.add("temblor");
        document.body.classList.add("arcoiris");

        mostrarMensajeGigante(
          "🚨 ¿POR QUÉ LO VOLVISTE A HACER? 🚨"
        );

        mensaje =
          "👽 ¡LA NASA HA PERDIDO EL CONTROL!";

      } else {

        // NIVEL 6 EN ADELANTE: CAOS TOTAL

        // Aumentar personajes hasta un máximo
        // de 120 para no sobrecargar el navegador.

        cantidad = Math.min(
          120,
          75 + (nivel - 5) * 10
        );

        personajes = [
          "🦆", "🐔", "🐥", "🐸",
          "🦄", "👽", "🛸", "💥",
          "🌈", "🦖", "🐙", "🍕",
          "🚀", "🧀", "🦕", "🎉"
        ];

        contenedor.classList.add("temblor");
        document.body.classList.add("arcoiris");

        const advertencias = [
          "🚨 ¡EL UNIVERSO ESTÁ EN PELIGRO!",
          "🦆 ¡LOS PATOS CONTROLAN INTERNET!",
          "👽 ¡HAN LLEGADO LOS ALIENS!",
          "🌍 ¡LA TIERRA HA SIDO REEMPLAZADA!",
          "🍕 ¡AHORA TODO ESTÁ HECHO DE PIZZA!",
          "💀 ¡ERROR 404: REALIDAD NO ENCONTRADA!"
        ];

        const indiceMensaje =
          (nivel - 6) % advertencias.length;

        mostrarMensajeGigante(
          advertencias[indiceMensaje]
        );

        mensaje =
          `💥 NIVEL DE CAOS ${nivel}: `
          + "¡ESTO YA ES IRREVERSIBLE!";
      }

      // Generar la invasión correspondiente
      crearLluvia(cantidad, personajes);

      // Mostrar descripción
      mensajeCaos.textContent = mensaje;

      // Detener efectos visuales
      setTimeout(() => {

        contenedor.classList.remove("temblor");

        document.body.classList.remove("arcoiris");

      }, 4500);

      // Permitir una nueva activación
      setTimeout(() => {

        botonProhibido.disabled = false;

        botonProhibido.textContent =
          "🚨 NO PRESIONAR OTRA VEZ";

      }, 4800);
    }


    // ------------------------------------------
    // EVENTO PRINCIPAL DEL BOTÓN PROHIBIDO
    // ------------------------------------------

    botonProhibido.addEventListener("click", () => {

      // Incrementar contador global de la página
      vecesPresionado++;

      contadorClicks.textContent = vecesPresionado;

      // Guardar el contador
      try {
        localStorage.setItem(
          "contadorCaos",
          String(vecesPresionado)
        );
      } catch (error) {
        // Continuar sin almacenamiento persistente
      }

      botonProhibido.disabled = true;

      let segundos = 3;

      mensajeCaos.textContent =
        "⚠️ INICIANDO PROTOCOLO DE AUTODESTRUCCIÓN...";

      // Cuenta regresiva dramática
      const cuentaRegresiva = setInterval(() => {

        mensajeCaos.textContent =
          `💣 DESASTRE EN ${segundos}...`;

        segundos--;

        if (segundos < 0) {

          clearInterval(cuentaRegresiva);

          // Ejecutar desastre correspondiente
          activarDesastre(vecesPresionado);
        }

      }, 800);

    });

  </script>

</body>
</html>


"""

dest = OUT_HTML_DIR / "boton.html"

dest.write_text(
    boton_html,
    encoding="utf-8"
)

print(f"Written: {dest.name}")
print(f"Size: {dest.stat().st_size:,} bytes")

Written: boton.html
Size: 18,414 bytes


## What `01_metadata_report.html` does

### File picker and FileReader

When the page loads, the `<input type="file">` element is visible but the output
area is hidden. Clicking the input opens the operating-system file picker.
After a JSON file is selected, the browser fires a `change` event.

The event handler creates a `FileReader` object and calls `reader.readAsText(file)`.
FileReader reads the local file directly — no HTTP request is made, and no server is
needed. When reading completes, the `reader.onload` callback receives
`evt.target.result`, which is the entire file content as a single UTF-8 string.

### JSON.parse and what it returns

`JSON.parse(text)` converts that string into a JavaScript object in memory.
If the text is not valid JSON — for example, if a non-JSON file is selected —
a `SyntaxError` is thrown and caught; the red error banner appears with the message.

After a successful parse, `loadDoc()` checks two things before rendering:

1. **Schema version** — `doc.schema_version === "1.0"`. The report only understands
   the schema produced by Phase 1. Any other value (or a missing key) shows an error.

2. **Required sections** — each of `metadata`, `assumptions`, `triangle`, `development`,
   and `ultimate_records` must be present and non-null. A missing section means the
   file is an incomplete export.

### DOM construction and what the browser shows

`renderMeta(doc)` unhides the `<div id="output">` and populates
`<div id="metaGrid">` — a two-column CSS grid. For each `[label, value]` pair
in the `rows` array, it creates two `<div>` elements with `createElement()` and
appends them to the grid. The CSS rule `grid-template-columns: 200px 1fr` places
labels in a fixed-width left column and values in a wider right column.

In the browser, the result is a clean definition list showing:
schema version, run ID, generated-at timestamp, cutoff, concept, method, basis,
units, selected average, tail convention, and the triangle dimensions
(84 accident periods × 84 development ages). If any field is `null`, an em-dash is
shown instead.


In [5]:
# 02 adds the triangle table and a toggle between cumulative and incremental views.
# The JavaScript builds the <table> using innerHTML string concatenation for speed.
# String labels from JSON (accident_period) are passed through escHtml() before
# insertion, so they cannot be interpreted as HTML markup by the browser.
html_02 = """\\
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>02 — Triangle Table Report</title>
  <style>
    body {
      font-family: system-ui, sans-serif;
      max-width: 1100px;
      margin: 40px auto;
      padding: 0 24px;
      color: #222;
      background: #fff;
    }
    h1 { color: #1a4c8c; margin-bottom: 4px; }

    .picker {
      display: flex; align-items: center; gap: 12px;
      margin: 24px 0; padding: 14px 16px;
      background: #f0f4fa; border-radius: 6px; border: 1px solid #d0daea;
    }
    .error-box {
      padding: 10px 14px; background: #fdf0f0;
      border-left: 4px solid #c0392b; color: #c0392b;
      border-radius: 0 4px 4px 0; margin-bottom: 16px;
    }

    /* Toggle button pair */
    .toggle-group { display: flex; margin-bottom: 12px; }
    .toggle-group button {
      padding: 6px 18px;
      border: 1px solid #1a4c8c;
      background: #fff; color: #1a4c8c;
      cursor: pointer; font-size: 14px;
    }
    .toggle-group button:first-child { border-radius: 4px 0 0 4px; }
    .toggle-group button:last-child  { border-radius: 0 4px 4px 0; border-left: none; }
    .toggle-group button.active { background: #1a4c8c; color: #fff; }

    /* Scrollable wrapper keeps the table from overflowing the page */
    .tri-scroll {
      overflow: auto;
      max-height: 460px;
      border: 1px solid #d0daea;
      border-radius: 4px;
    }

    /* Development triangle table */
    .tri { border-collapse: collapse; font-size: 11px; white-space: nowrap; }
    /* Sticky header row */
    .tri thead th {
      position: sticky; top: 0; z-index: 1;
      background: #1a4c8c; color: #fff;
      padding: 4px 8px; border: 1px solid #14397a; text-align: center;
    }
    /* Sticky first column (accident-period label) */
    .tri thead th:first-child,
    .tri tbody td:first-child {
      position: sticky; left: 0; z-index: 2;
    }
    .tri thead th:first-child { z-index: 3; }
    .tri tbody td:first-child {
      background: #f0f4fa; font-weight: 600;
      text-align: left; border-right: 2px solid #1a4c8c;
      padding: 3px 10px;
    }
    /* Data cells */
    .tri tbody td {
      padding: 3px 7px; text-align: right;
      border: 1px solid #e0e8f0; min-width: 68px;
    }
    /* null = not yet observed at this cutoff; shown as grey */
    .tri tbody td.null-cell { background: #f5f5f5; color: #bbb; text-align: center; }
  </style>
</head>
<body>
  <h1>02 — Triangle Table Report</h1>
  <p>Load a <code>teaching_export_*.json</code> to view the development triangle.</p>

  <div class="picker">
    <label for="fileInput"><strong>JSON file:</strong></label>
    <input type="file" id="fileInput" accept=".json">
  </div>

  <div id="errBox" class="error-box" style="display:none"></div>
  <div id="output"                   style="display:none">
    <h2 id="triTitle">Cumulative triangle</h2>

    <!-- JavaScript switches between the two data arrays on button click -->
    <div class="toggle-group">
      <button id="btnC" class="active" onclick="setMode('cumulative')">Cumulative</button>
      <button id="btnI"               onclick="setMode('incremental')">Incremental</button>
    </div>

    <div class="tri-scroll">
      <table class="tri" id="triTable"></table>
    </div>
  </div>

  <script>
    var _doc  = null;           // parsed JSON document
    var _mode = 'cumulative';   // current display mode

    document.getElementById('fileInput').addEventListener('change', function (e) {
      var file = e.target.files[0];
      if (!file) return;
      var reader = new FileReader();
      reader.onload = function (evt) { loadDoc(evt.target.result); };
      reader.readAsText(file);
    });

    function showError(msg) {
      document.getElementById('errBox').textContent = msg;
      document.getElementById('errBox').style.display = '';
      document.getElementById('output').style.display = 'none';
    }

    // escHtml() prevents JSON string values from being interpreted as HTML markup
    // when inserted into an innerHTML string.  Numbers from fmt() are always safe
    // (digits, comma, em-dash only), so only string labels need this guard.
    function escHtml(s) {
      return String(s).replace(/[&<>"']/g, function (c) {
        return {'&': '&amp;', '<': '&lt;', '>': '&gt;', '"': '&quot;', "'": '&#39;'}[c];
      });
    }

    function loadDoc(text) {
      // JSON.parse() throws SyntaxError if the text is not valid JSON.
      var doc;
      try { doc = JSON.parse(text); }
      catch (e) { showError('Invalid JSON: ' + e.message); return; }

      // Enforce schema version; only "1.0" is supported by this report.
      if (doc.schema_version !== '1.0') {
        showError('Unsupported schema_version: ' + JSON.stringify(doc.schema_version));
        return;
      }

      // Verify every required top-level section exists.
      var required = ['metadata', 'assumptions', 'triangle', 'development', 'ultimate_records'];
      for (var i = 0; i < required.length; i++) {
        if (doc[required[i]] == null) { showError('Missing: ' + required[i]); return; }
      }

      // Verify the triangle has the arrays this report depends on.
      var t = doc.triangle;
      if (!Array.isArray(t.accident_periods) || !Array.isArray(t.development_ages)) {
        showError('triangle.accident_periods or triangle.development_ages is missing or not an array');
        return;
      }
      if (!Array.isArray(t.cumulative) || !Array.isArray(t.incremental)) {
        showError('triangle.cumulative or triangle.incremental is missing or not an array');
        return;
      }

      _doc = doc;
      document.getElementById('errBox').style.display = 'none';
      document.getElementById('output').style.display = '';
      // Wrap the render call so any unexpected data shape shows an error banner
      // rather than leaving a blank, silent page.
      try {
        renderTriangle('cumulative');
      } catch (e) {
        showError('Render error: ' + e.message);
      }
    }

    function setMode(mode) {
      _mode = mode;
      document.getElementById('btnC').className = (mode === 'cumulative') ? 'active' : '';
      document.getElementById('btnI').className = (mode === 'incremental') ? 'active' : '';
      try {
        renderTriangle(mode);
      } catch (e) {
        showError('Render error: ' + e.message);
      }
    }

    function fmt(v) {
      // Format a number with thousand separators, no decimals; null shows as an em-dash.
      if (v == null) return '—';
      return v.toLocaleString(undefined, {maximumFractionDigits: 0});
    }

    function renderTriangle(mode) {
      if (!_doc) return;
      var t    = _doc.triangle;
      // Select the 2D values array: rows are accident periods, columns are development ages.
      var data = (mode === 'cumulative') ? t.cumulative : t.incremental;
      var aps  = t.accident_periods;   // row labels (strings from JSON)
      var ages = t.development_ages;   // column headers (integers from JSON)

      document.getElementById('triTitle').textContent =
        (mode === 'cumulative' ? 'Cumulative' : 'Incremental') + ' triangle';

      // Build the table as an HTML string; innerHTML assignment is faster
      // for large tables than creating each cell with createElement().
      // String labels from JSON are passed through escHtml() so that a label
      // containing '<' or '>' cannot be interpreted as HTML markup.
      // Numeric values returned by fmt() are safe without escaping (digits,
      // comma separators, and the em-dash only).
      var html = '<thead><tr><th>Accident period</th>';
      for (var j = 0; j < ages.length; j++) {
        // ages[j] is an integer; no escaping needed.
        html += '<th>dev ' + ages[j] + '</th>';
      }
      html += '</tr></thead><tbody>';

      for (var i = 0; i < aps.length; i++) {
        // aps[i] is a string label from JSON; escHtml() prevents markup injection.
        html += '<tr><td>' + escHtml(aps[i]) + '</td>';
        for (var j = 0; j < ages.length; j++) {
          var v = data[i][j];
          if (v == null) {
            // null = cell not observed at this cutoff; grey styling distinguishes it from zero.
            html += '<td class="null-cell">—</td>';
          } else {
            html += '<td>' + fmt(v) + '</td>';
          }
        }
        html += '</tr>';
      }
      html += '</tbody>';
      document.getElementById('triTable').innerHTML = html;
    }
  </script>
</body>
</html>
"""

dest = OUT_HTML_DIR / '02_triangle_table_report.html'
dest.write_text(html_02, encoding='utf-8')
print(f'Written: {dest.name}  ({dest.stat().st_size:,} bytes)')


Written: 02_triangle_table_report.html  (8,792 bytes)


## What `02_triangle_table_report.html` adds

### Two-button toggle

Two `<button>` elements sit inside a `.toggle-group` flex container.
Each button's `onclick` attribute calls `setMode()` with either `'cumulative'`
or `'incremental'`. `setMode()` updates the `class` attribute of both buttons
so that only the active one gets the `.active` CSS style (dark blue background,
white text), then calls `renderTriangle(mode)`.

### How the 2D array becomes an HTML table

`doc.triangle.cumulative` and `doc.triangle.incremental` are JavaScript arrays
where `data[i][j]` is the value for accident-period row `i` and
development-age column `j`. The triangle has 84 rows and 84 columns.

`renderTriangle()` builds the table as one HTML string using `+=` concatenation
and then assigns it to `document.getElementById('triTable').innerHTML`.
This single DOM assignment is significantly faster than calling
`document.createElement('td')` 7,056 times.

### Safe label rendering with `escHtml()`

`accident_period` values in the JSON are strings. Inserting a string directly into an
HTML string with `+` concatenation can be unsafe: a label containing `<` or `>` would
be interpreted as markup by the browser. `escHtml()` converts the five special HTML
characters (`&`, `<`, `>`, `"`, `'`) to their entity equivalents (`&amp;`, `&lt;`,
`&gt;`, `&quot;`, `&#39;`) before insertion, so labels are always rendered as
plain text. Numeric values from `fmt()` contain only digits, commas, and the em-dash,
so they do not need escaping.

The header row iterates `t.development_ages` to produce one `<th>dev N</th>` per age.
The body iterates `t.accident_periods` for row labels, then `t.development_ages`
for cells in each row.

### How null cells are handled

JSON `null` represents a cell that was not observed before the cutoff date — it is
structurally different from a zero value. The condition `data[i][j] == null` is
`true` for both `null` and `undefined` in JavaScript. Matching cells receive
`class="null-cell"` which the CSS styles as light grey with a centred em-dash.

### CSS sticky positioning

`position: sticky; top: 0` on `thead th` keeps the header row pinned while
scrolling vertically through 84 accident periods.
`position: sticky; left: 0` on the first column (`td:first-child`) keeps the
accident-period label pinned while scrolling horizontally through 84 development ages.
The `.tri-scroll` container has `overflow: auto` and `max-height: 460px` to
keep the table within the viewport.

### What the browser shows

A scrollable table with a blue sticky header row. Observed cells show formatted
integers; the upper-right portion of the table (future development) is all grey
em-dash cells. Clicking "Incremental" re-renders the same table with incremental
values; toggling back to "Cumulative" re-renders with cumulative values.


In [6]:
# 03 adds a heatmap (white-to-blue colour scale) and an ultimate-estimates table.
# The heatRgb function uses linear interpolation between two RGB triples.
# escHtml() guards all string labels from JSON before innerHTML insertion.
# Null cells (not yet observed at the cutoff) get a grey background, not heat colour.
html_03 = """\\
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>03 — Interactive Triangle Report</title>
  <style>
    body {
      font-family: system-ui, sans-serif;
      max-width: 1100px;
      margin: 40px auto;
      padding: 0 24px;
      color: #222;
      background: #fff;
    }
    h1, h2 { color: #1a4c8c; }
    h2 { margin-top: 32px; }

    .picker {
      display: flex; align-items: center; gap: 12px;
      margin: 24px 0; padding: 14px 16px;
      background: #f0f4fa; border-radius: 6px; border: 1px solid #d0daea;
    }
    .error-box {
      padding: 10px 14px; background: #fdf0f0;
      border-left: 4px solid #c0392b; color: #c0392b;
      border-radius: 0 4px 4px 0; margin-bottom: 16px;
    }

    .toggle-group { display: flex; margin-bottom: 10px; }
    .toggle-group button {
      padding: 6px 18px;
      border: 1px solid #1a4c8c;
      background: #fff; color: #1a4c8c;
      cursor: pointer; font-size: 14px;
    }
    .toggle-group button:first-child { border-radius: 4px 0 0 4px; }
    .toggle-group button:last-child  { border-radius: 0 4px 4px 0; border-left: none; }
    .toggle-group button.active { background: #1a4c8c; color: #fff; }

    /* Colour legend below the toggle */
    .legend {
      display: flex; align-items: center; gap: 10px;
      margin-bottom: 12px; font-size: 12px; color: #555;
    }
    .legend-bar {
      width: 180px; height: 14px;
      border: 1px solid #d0daea; border-radius: 3px;
    }

    .tri-scroll { overflow: auto; max-height: 460px; border: 1px solid #d0daea; border-radius: 4px; }
    .tri { border-collapse: collapse; font-size: 11px; white-space: nowrap; }
    .tri thead th {
      position: sticky; top: 0; z-index: 1;
      background: #1a4c8c; color: #fff;
      padding: 4px 8px; border: 1px solid #14397a; text-align: center;
    }
    .tri thead th:first-child { position: sticky; left: 0; z-index: 3; }
    .tri tbody td:first-child {
      position: sticky; left: 0; z-index: 2;
      background: #f0f4fa; font-weight: 600;
      text-align: left; border-right: 2px solid #1a4c8c;
      padding: 3px 10px;
    }
    .tri tbody td {
      padding: 3px 7px; text-align: right;
      border: 1px solid #e0e8f0; min-width: 68px;
    }
    .tri tbody td.null-cell { background: #f5f5f5; color: #bbb; text-align: center; }

    /* Ultimate table */
    .ult-wrap { overflow-x: auto; margin-top: 8px; }
    .ult { border-collapse: collapse; font-size: 12px; width: 100%; }
    .ult thead th {
      background: #1a4c8c; color: #fff;
      padding: 5px 12px; text-align: left;
      font-weight: 600; border: 1px solid #14397a;
    }
    .ult tbody td { padding: 4px 12px; border-bottom: 1px solid #e8eef5; }
    .ult tbody tr:hover td { background: #f5f8fd; }
    .ult tbody td.num { text-align: right; font-family: monospace; }
    /* Highlight the ultimate column */
    .ult tbody td.ult-col { font-weight: 600; }
  </style>
</head>
<body>
  <h1>03 — Interactive Triangle Report</h1>
  <p>Load a <code>teaching_export_*.json</code> to view the heatmap triangle and ultimate estimates.</p>

  <div class="picker">
    <label for="fileInput"><strong>JSON file:</strong></label>
    <input type="file" id="fileInput" accept=".json">
  </div>

  <div id="errBox" class="error-box" style="display:none"></div>
  <div id="output"                   style="display:none">

    <!-- ── Triangle section ───────────────────────────────── -->
    <h2>Development triangle</h2>

    <div class="toggle-group">
      <button id="btnC" class="active" onclick="setMode('cumulative')">Cumulative</button>
      <button id="btnI"               onclick="setMode('incremental')">Incremental</button>
    </div>

    <!-- Legend: shows the colour scale from low (left) to high (right) -->
    <div class="legend">
      <span>Low</span>
      <!-- background is set by renderLegend() after the range is known -->
      <span class="legend-bar" id="legendBar"></span>
      <span>High</span>
      <span id="legendRange"></span>
    </div>

    <div class="tri-scroll">
      <table class="tri" id="triTable"></table>
    </div>

    <!-- ── Ultimate estimates section ────────────────── -->
    <h2>Ultimate estimates</h2>
    <div class="ult-wrap">
      <table class="ult" id="ultTable"></table>
    </div>
  </div>

  <script>
    var _doc  = null;
    var _mode = 'cumulative';

    document.getElementById('fileInput').addEventListener('change', function (e) {
      var file = e.target.files[0];
      if (!file) return;
      var reader = new FileReader();
      reader.onload = function (evt) { loadDoc(evt.target.result); };
      reader.readAsText(file);
    });

    function showError(msg) {
      document.getElementById('errBox').textContent = msg;
      document.getElementById('errBox').style.display = '';
      document.getElementById('output').style.display = 'none';
    }

    // escHtml() prevents JSON string values from being interpreted as HTML markup
    // when inserted into an innerHTML string.  Numbers from fmt*() are always safe
    // (digits, comma, em-dash, percent only), so only string labels need this guard.
    function escHtml(s) {
      return String(s).replace(/[&<>"']/g, function (c) {
        return {'&': '&amp;', '<': '&lt;', '>': '&gt;', '"': '&quot;', "'": '&#39;'}[c];
      });
    }

    function loadDoc(text) {
      var doc;
      try { doc = JSON.parse(text); }
      catch (e) { showError('Invalid JSON: ' + e.message); return; }

      if (doc.schema_version !== '1.0') {
        showError('Unsupported schema_version: ' + JSON.stringify(doc.schema_version));
        return;
      }
      var required = ['metadata', 'assumptions', 'triangle', 'development', 'ultimate_records'];
      for (var i = 0; i < required.length; i++) {
        if (doc[required[i]] == null) { showError('Missing: ' + required[i]); return; }
      }

      // Verify the triangle has the arrays this report depends on.
      var t = doc.triangle;
      if (!Array.isArray(t.accident_periods) || !Array.isArray(t.development_ages)) {
        showError('triangle.accident_periods or triangle.development_ages is missing or not an array');
        return;
      }
      if (!Array.isArray(t.cumulative) || !Array.isArray(t.incremental)) {
        showError('triangle.cumulative or triangle.incremental is missing or not an array');
        return;
      }
      if (!Array.isArray(doc.ultimate_records)) {
        showError('ultimate_records is not an array');
        return;
      }

      _doc = doc;
      document.getElementById('errBox').style.display = 'none';
      document.getElementById('output').style.display = '';
      // Wrap the render calls so any unexpected data shape shows an error banner
      // rather than leaving a blank, silent page.
      try {
        renderTriangle('cumulative');
        renderUltimate();
      } catch (e) {
        showError('Render error: ' + e.message);
      }
    }

    function setMode(mode) {
      _mode = mode;
      document.getElementById('btnC').className = (mode === 'cumulative') ? 'active' : '';
      document.getElementById('btnI').className = (mode === 'incremental') ? 'active' : '';
      try {
        renderTriangle(mode);
      } catch (e) {
        showError('Render error: ' + e.message);
      }
    }

    // ── Colour helpers ────────────────────────────────────────────────

    // Interpolate between two RGB triples; t in [0, 1].
    function lerp(a, b, t) {
      return [
        Math.round(a[0] + t * (b[0] - a[0])),
        Math.round(a[1] + t * (b[1] - a[1])),
        Math.round(a[2] + t * (b[2] - a[2])),
      ];
    }

    // COL_MIN = white: anchors the minimum observed value (not necessarily literal zero).
    var COL_MIN = [255, 255, 255];
    var COL_MAX = [26,  76,  140];
    var COL_NEG = [192, 57,  43];   // red for negative incremental values (not in supplied export)

    function heatRgb(v, vmin, vmax) {
      // Map v to a colour on the white-to-blue (or white-to-red for negatives) scale.
      // With the supplied export vmin >= 0 always holds, so only the first branch fires.
      if (vmin >= 0) {
        var t = (vmax > vmin) ? (v - vmin) / (vmax - vmin) : 0;
        t = Math.max(0, Math.min(1, t));
        return lerp(COL_MIN, COL_MAX, t);
      }
      // Mixed range (vmin < 0): negative values shade towards red, positive towards blue.
      if (v < 0) {
        var t = (vmin < 0) ? Math.abs(v) / Math.abs(vmin) : 0;
        t = Math.max(0, Math.min(1, t));
        return lerp(COL_MIN, COL_NEG, t);
      }
      var t = (vmax > 0) ? v / vmax : 0;
      t = Math.max(0, Math.min(1, t));
      return lerp(COL_MIN, COL_MAX, t);
    }

    function rgbStr(rgb) { return 'rgb(' + rgb[0] + ',' + rgb[1] + ',' + rgb[2] + ')'; }

    function contrastColor(rgb) {
      // BT.601 luminance: returns black (#111) for light backgrounds, white for dark.
      var lum = (0.299 * rgb[0] + 0.587 * rgb[1] + 0.114 * rgb[2]) / 255;
      return (lum > 0.5) ? '#111' : '#fff';
    }

    // ── Triangle rendering ────────────────────────────────────────────

    function flatValues(data) {
      // Collect all non-null values into a flat array for range calculation.
      var vals = [];
      for (var i = 0; i < data.length; i++) {
        for (var j = 0; j < data[i].length; j++) {
          if (data[i][j] != null) vals.push(data[i][j]);
        }
      }
      return vals;
    }

    function renderLegend(vmin, vmax) {
      var bar = document.getElementById('legendBar');
      if (vmin >= 0) {
        bar.style.background = 'linear-gradient(to right, ' +
          rgbStr(COL_MIN) + ', ' + rgbStr(COL_MAX) + ')';
      } else {
        // Three-stop gradient: red (most negative) → white (zero) → blue (most positive).
        // Note: white appears at the 50% mark; zero's true position is |vmin|/(|vmin|+vmax).
        bar.style.background = 'linear-gradient(to right, ' +
          rgbStr(COL_NEG) + ', ' + rgbStr(COL_MIN) + ', ' + rgbStr(COL_MAX) + ')';
      }
      var rng = document.getElementById('legendRange');
      rng.textContent = '(' + fmt(vmin) + ' to ' + fmt(vmax) + ')';
    }

    function fmt(v) {
      if (v == null) return '—';
      return v.toLocaleString(undefined, {maximumFractionDigits: 0});
    }

    function renderTriangle(mode) {
      if (!_doc) return;
      var t    = _doc.triangle;
      var data = (mode === 'cumulative') ? t.cumulative : t.incremental;
      var aps  = t.accident_periods;
      var ages = t.development_ages;

      // Calculate value range for normalisation.
      var vals = flatValues(data);
      var vmin = vals.length ? Math.min.apply(null, vals) : 0;
      var vmax = vals.length ? Math.max.apply(null, vals) : 0;
      renderLegend(vmin, vmax);

      // Build the table as an HTML string for speed (one DOM assignment for 7,056 cells).
      // String labels from JSON (aps[i]) are passed through escHtml() so that a label
      // containing '<' or '>' cannot be interpreted as HTML markup.
      // ages[j] is an integer; fmt() returns digits/comma/em-dash only: both are safe.
      var html = '<thead><tr><th>Accident period</th>';
      for (var j = 0; j < ages.length; j++) html += '<th>dev ' + ages[j] + '</th>';
      html += '</tr></thead><tbody>';

      for (var i = 0; i < aps.length; i++) {
        html += '<tr><td>' + escHtml(aps[i]) + '</td>';
        for (var j = 0; j < ages.length; j++) {
          var v = data[i][j];
          if (v == null) {
            // Not observed: grey cell, no heat applied.
            html += '<td class="null-cell">—</td>';
          } else {
            // Compute background colour and a readable foreground.
            var rgb = heatRgb(v, vmin, vmax);
            var bg  = rgbStr(rgb);
            var fg  = contrastColor(rgb);
            html += '<td style="background:' + bg + ';color:' + fg + '">' + fmt(v) + '</td>';
          }
        }
        html += '</tr>';
      }
      html += '</tbody>';
      document.getElementById('triTable').innerHTML = html;
    }

    // ── Ultimate estimates ────────────────────────────────────────────────

    function renderUltimate() {
      if (!_doc) return;
      var recs = _doc.ultimate_records;

      // Column definitions: [header, accessor function, extra CSS class].
      // Each accessor returns a pre-formatted string, so the value is safe to
      // insert as-is (digits, comma, percent, em-dash only).
      // The accident_period label is a string from JSON and is passed through
      // escHtml() to prevent markup injection.
      var cols = [
        ['Accident period',        function (r) { return escHtml(r.accident_period); },    ''],
        ['Current incurred',       function (r) { return fmtNum(r.current_incurred); },    'num'],
        ['ATU',                    function (r) { return fmtDec(r.atu, 4); },              'num'],
        ['% Developed',            function (r) { return fmtPct(r.percent_developed); },   'num'],
        ['Ultimate',               function (r) { return fmtNum(r.ultimate); },            'num ult-col'],
        ['Additional development', function (r) { return fmtNum(r.additional_development); }, 'num'],
      ];

      var html = '<thead><tr>';
      for (var k = 0; k < cols.length; k++) html += '<th>' + escHtml(cols[k][0]) + '</th>';
      html += '</tr></thead><tbody>';

      for (var i = 0; i < recs.length; i++) {
        var r = recs[i];
        html += '<tr>';
        for (var k = 0; k < cols.length; k++) {
          var cls = cols[k][2] ? ' class="' + cols[k][2] + '"' : '';
          var val = cols[k][1](r);
          html += '<td' + cls + '>' + (val !== '' && val != null ? val : '—') + '</td>';
        }
        html += '</tr>';
      }
      html += '</tbody>';
      document.getElementById('ultTable').innerHTML = html;
    }

    // Number formatters — return strings containing only digits, commas, percent, and em-dash.
    function fmtNum(v) {
      if (v == null) return '—';
      return v.toLocaleString(undefined, {minimumFractionDigits: 0, maximumFractionDigits: 0});
    }
    function fmtDec(v, dp) {
      if (v == null) return '—';
      return v.toFixed(dp);
    }
    function fmtPct(v) {
      if (v == null) return '—';
      return (v * 100).toFixed(1) + '%';
    }
  </script>
</body>
</html>
"""

dest = OUT_HTML_DIR / '03_interactive_triangle_report.html'
dest.write_text(html_03, encoding='utf-8')
print(f'Written: {dest.name}  ({dest.stat().st_size:,} bytes)')


Written: 03_interactive_triangle_report.html  (15,175 bytes)


## What `03_interactive_triangle_report.html` adds

### Linear colour interpolation

`heatRgb(v, vmin, vmax)` maps a cell value to an RGB colour tuple.
For a non-negative range it computes a position `t` in `[0, 1]`:

```
t = (v − vmin) / (vmax − vmin)
```

and interpolates between white `(255, 255, 255)` (at `t = 0`) and
dark blue `(26, 76, 140)` (at `t = 1`) using `lerp()`:

```
R = round(255 + t × (26  − 255))   →  255 at the minimum, 26 at the maximum
G = round(255 + t × (76  − 255))
B = round(255 + t × (140 − 255))
```

When the incremental view is active, `vmin` can be negative if the export contains negative development (reserve reductions). In that case, values below zero interpolate from white towards red `(192, 57, 43)`, and values above zero interpolate from white towards blue. `Math.max(0, Math.min(1, t))` clamps `t` so rounding errors in the extremes never produce out-of-range RGB values.

> **Note on the supplied export.** `teaching_export_2022-12.json` contains no negative incremental values, so `vmin >= 0` always holds and the red colour branch is not activated. The legend renders a two-stop white-to-blue gradient. The red branch is present in the code for use with any export that does contain negative development.

### Readable text on any background colour

After `heatRgb()` computes the background colour, `contrastColor(rgb)` applies
the BT.601 perceptual luminance formula:

```
luminance = (0.299 × R + 0.587 × G + 0.114 × B) / 255
```

BT.601 weights the green channel most heavily because the human eye is most
sensitive to green. If `luminance > 0.5` the background is light (close to white)
and dark text `#111` is returned. Otherwise white text `#fff` is returned.
The cell's `style` attribute receives both background and color:
`style="background:rgb(R,G,B);color:#111"`.

### Legend bar and dynamic range

`flatValues(data)` collects all non-null cell values into a flat array.
`Math.min.apply(null, vals)` and `Math.max.apply(null, vals)` find the range.

`renderLegend()` sets the `background` CSS property of `<span id="legendBar">` to
`linear-gradient(to right, rgb(…zero…), rgb(…max…))` — the same colours the heatmap
uses, so the legend is always consistent. The `<span id="legendRange">` shows
the numeric `(vmin – vmax)` interval. Both update on every `renderTriangle()` call,
so switching between cumulative and incremental automatically updates the legend to
reflect the new range.

### Ultimate records table

`renderUltimate()` iterates `doc.ultimate_records`, which contains one object per
accident period with keys `accident_period`, `current_incurred`, `atu`,
`percent_developed`, `ultimate`, and `additional_development`.

The `cols` array holds column definitions as `[header, accessorFn, cssClass]` triples.
The outer loop iterates records; the inner loop iterates column definitions and calls
the accessor function for each cell.

The `accident_period` accessor wraps its value in `escHtml()` — the only string label
in the table. All other accessors return strings from `fmtNum`, `fmtDec`, or `fmtPct`,
which contain only digits, punctuation, and the percent sign, and are safe without escaping. This separation means the table-building loop
does not need to be rewritten when a column is added or reordered.

`fmtNum()` formats values with `toLocaleString()` using zero decimal places.
`fmtDec(v, 4)` formats the ATU factor to four decimal places.
`fmtPct(v)` multiplies by 100 and appends `%`.

### What the browser shows

The triangle now has coloured cells: the oldest, most-developed accident periods
in the lower-left corner are the darkest blue (highest cumulative values);
recent periods with few development ages are pale. With the supplied export the
incremental view is also white-to-blue (no negative values are present). The legend
bar below the toggle shows the gradient and updates its range each time the mode
switches. Hovering over the ultimate table highlights rows in pale blue.


## Comparison: this learning report vs `teaching_viewer.html`

| Feature | Learning report (this notebook) | Production viewer |
|---|---|---|
| Lines of code | ~100 / ~200 / ~300 per artifact | ~930 |
| Triangle display | HTML `<table>` built with `innerHTML` string concatenation | `createElement` + `textContent` per cell; defines `escText()` to HTML-escape values before rendering |
| Colour scale | Fixed white-to-blue | CSS custom properties; dark-mode aware |
| Dark mode | Not implemented | `@media (prefers-color-scheme: dark)` tokens |
| Theme switch | No | `matchMedia` listener re-renders on OS switch |
| Multiple views | Cumulative / Incremental toggle | Cumulative / Incremental + Factors tabs |
| ATA / ATU display | Not included | Separate factor tab |
| Validation | schema_version + required keys | Same + per-cell type checks |
| Offline | Yes (no CDN, no server) | Yes |
| External dependencies | None | None |

The learning report deliberately omits:
- the factor tables (ATA, ATU)
- dark-mode CSS tokens
- the validation viewer tab
- the sticky-header band navigation

These are left out so each concept can be introduced individually.
The production viewer applies all of these patterns simultaneously — reading it
after this notebook should make its structure clear.


In [7]:
import re

# Confirm each artifact exists, has a plausible size, and contains no external URLs.
artifacts = [
    '01_metadata_report.html',
    '02_triangle_table_report.html',
    '03_interactive_triangle_report.html',
]

all_ok = True
for name in artifacts:
    p = OUT_HTML_DIR / name
    if not p.exists():
        print(f'MISSING : {name}')
        all_ok = False
        continue
    content = p.read_text(encoding='utf-8')
    # Strip HTML comments before checking for URLs so that explanatory links
    # inside comment blocks do not trigger a false positive.
    no_comments = re.sub(r'<!--.*?-->', '', content, flags=re.DOTALL)
    ext_urls = re.findall(r'https?://', no_comments)
    if ext_urls:
        print(f'EXT-URL : {name} — found {len(ext_urls)} external URL(s)')
        all_ok = False
    else:
        print(f'OK      : {name}  {p.stat().st_size:,} bytes  no external URLs')

print()
print('All artifacts OK.' if all_ok else 'One or more checks FAILED.')
print()
print('Manual browser check:')
print('  Open each HTML file locally and select teaching_export_2022-12.json.')
print('  01: verify metadata grid loads without errors.')
print('  02: verify triangle table renders; toggle switches between cumulative/incremental.')
print('  03: verify heatmap colours cells; ultimate table shows all 84 accident periods.')


OK      : 01_metadata_report.html  5,687 bytes  no external URLs
OK      : 02_triangle_table_report.html  8,792 bytes  no external URLs
OK      : 03_interactive_triangle_report.html  15,175 bytes  no external URLs

All artifacts OK.

Manual browser check:
  Open each HTML file locally and select teaching_export_2022-12.json.
  01: verify metadata grid loads without errors.
  02: verify triangle table renders; toggle switches between cumulative/incremental.
  03: verify heatmap colours cells; ultimate table shows all 84 accident periods.
